# MMS (facebook/mms-1b-all, қазақша `kaz` adapter) + VAD — 100 аудио

CTC моделі: Whisper-дан өзгеше, цикл/hallucination жасамайды, тыныс белгісі мен бас әріп жоқ.
Нәтиже `dataset_wav16k/mms/` қалтасына сақталады. Әр файлдан кейін CSV-ге сақталады (resume-safe).
GPU (T4) қосыңыз: Runtime → Change runtime type → T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pydub jiwer sacrebleu bert_score transformers accelerate

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.8 MB/s eta 0:00:00


In [ ]:
import os, time, re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pydub import AudioSegment

DATASET_BASE = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k"
OUT_DIR = os.path.join(DATASET_BASE, "mms")
os.makedirs(OUT_DIR, exist_ok=True)
DURATIONS = ["30s", "1m", "2m", "5m", "10m"]

def collect_pairs(base, durations):
    pairs = []
    for d in durations:
        audio_dir = os.path.join(base, "audio", d)
        ref_dir = os.path.join(base, "references", d)
        if not os.path.isdir(audio_dir):
            print("[WARN] аудио қалтасы жоқ:", audio_dir); continue
        for fname in sorted(os.listdir(audio_dir)):
            if not fname.lower().endswith(".wav"): continue
            stem = os.path.splitext(fname)[0]
            ref_path = os.path.join(ref_dir, stem + ".txt")
            if not os.path.exists(ref_path):
                print("[WARN] REF жоқ:", ref_path); continue
            pairs.append(dict(duration=d, stem=stem,
                              audio_path=os.path.join(audio_dir, fname), ref_path=ref_path))
    return pairs

pairs = collect_pairs(DATASET_BASE, DURATIONS)
print("Жұп саны:", len(pairs))

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Жұп саны: 100


In [ ]:
import jiwer, sacrebleu, bert_score

def norm_text(t):
    t = str(t).lower()
    t = re.sub(r"[^\w\s]", "", t)
    return re.sub(r"\s+", " ", t).strip()

COMPUTE_TER = True   # ұзын (5m/10m) файлдарда TER баяу (~1 мин/файл). Қаламасаңыз False қойыңыз.

_scorer = bert_score.BERTScorer(lang="other", model_type="bert-base-multilingual-cased")  # бір рет жүктеледі
_ter = sacrebleu.metrics.TER()

def all_metrics(ref, hyp, with_ter=True, with_bert=True):
    r, h = norm_text(ref), norm_text(hyp)
    out = dict(
        wer=jiwer.wer(r, h), cer=jiwer.cer(r, h),
        bleu=sacrebleu.sentence_bleu(h, [r]).score,
        chrf=sacrebleu.sentence_chrf(h, [r]).score,
    )
    out["ter"] = _ter.sentence_score(h, [r]).score if (with_ter and COMPUTE_TER) else float("nan")
    out["bertscore"] = _scorer.score([h], [r])[2].item() if with_bert else float("nan")
    return out

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, AutoProcessor

MMS_MODEL = "facebook/mms-1b-all"
TARGET_LANG = "kaz"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MMS_MODEL, target_lang=TARGET_LANG)
mms = Wav2Vec2ForCTC.from_pretrained(MMS_MODEL, target_lang=TARGET_LANG, ignore_mismatched_sizes=True)
mms = mms.to(device).eval()
if device == "cuda":
    mms = mms.half()
print("Жүктелді:", MMS_MODEL, "| lang:", TARGET_LANG, "| device:", device)

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.kaz.safetensors: reconstructing file:   0%|          |  0.00B / 9.19MB            

adapter.kaz.safetensors: downloading bytes:           |  0.00B            

Жүктелді: facebook/mms-1b-all | lang: kaz | device: cuda


In [ ]:
def vad_segment(x, sr=16000, win=0.025, min_sil=0.30, max_len=18.0, min_len=1.0, pad=0.25):
    n = int(win * sr)
    if len(x) < n * 2:
        return [(0, len(x))]
    frames = x[: len(x) // n * n].reshape(-1, n)
    e = np.sqrt((frames ** 2).mean(axis=1) + 1e-12)
    floor = np.percentile(e, 10); speech = np.percentile(e, 75)
    thr = floor + 0.12 * max(speech - floor, 1e-6)
    sil = e < thr
    need = max(1, int(min_sil / win))
    cuts, run = [], 0
    for i, s in enumerate(sil):
        if s: run += 1
        else:
            if run >= need: cuts.append((i - run // 2) * n)
            run = 0
    bounds = sorted({0, len(x)} | {c for c in cuts if 0 < c < len(x)})
    segs, a = [], bounds[0]
    for b in bounds[1:]:
        if (b - a) / sr < min_len and b != bounds[-1]:
            continue
        while (b - a) / sr > max_len:
            segs.append((a, a + int(max_len * sr))); a += int(max_len * sr)
        segs.append((a, b)); a = b
    return [(max(0, s - int(pad * sr)), min(len(x), e2 + int(pad * sr))) for s, e2 in segs]

def load_audio(path, sr=16000):
    a = AudioSegment.from_file(path).set_channels(1).set_frame_rate(sr)
    x = np.array(a.get_array_of_samples()).astype(np.float32)
    return x / float(1 << (8 * a.sample_width - 1)), sr

@torch.no_grad()
def mms_transcribe_array(chunk, sr=16000):
    inputs = processor(chunk, sampling_rate=sr, return_tensors="pt")
    iv = inputs["input_values"].to(device)
    if device == "cuda":
        iv = iv.half()
    ids = torch.argmax(mms(iv).logits, dim=-1)[0]
    return processor.decode(ids).strip()

def transcribe_mms_vad(audio_path):
    x, sr = load_audio(audio_path)
    texts = []
    for s, e in vad_segment(x, sr):
        c = x[s:e]
        if len(c) < sr * 0.2: continue
        texts.append(mms_transcribe_array(c, sr))
    return " ".join(t for t in texts if t)

In [ ]:
# Тест: бір файл
p = pairs[0]
t0 = time.time()
test_raw = transcribe_mms_vad(p["audio_path"])
ref0 = open(p["ref_path"], encoding="utf-8").read().strip()
print("уақыт:", round(time.time()-t0, 1), "сек")
print("REF:", ref0[:200]); print("MMS:", test_raw[:200])
print(all_metrics(ref0, test_raw, with_ter=False, with_bert=False))

уақыт: 6.6 сек
REF: Түптеп келген кезде, бәрі мәдениеттен басталады тағыда. Тамақтану мәдениеті, чек-ап жасау жылына бір рет дедіңіз ба, жарты жылда бір рет па? Жылына бір рет жеткілікті. Жылына бір рет чек-ап жасап, орг
MMS: түптеп келген кезсе бәрі мәдениеттен басталаттағы да тамақтану мәдениеті чекап жасау жылына бір реттедіңіз бар жарты жылда бір жылдана брет жеткіледі жылына біржет чекап жасап организмде не болып жаты
{'wer': 0.3424657534246575, 'cer': 0.12681912681912683, 'bleu': 44.907071012275345, 'chrf': 83.76703094248758, 'ter': nan, 'bertscore': nan}


In [ ]:
OUT_CSV = os.path.join(OUT_DIR, "mapp_v2_batch_results_mms_vad.csv")
done, results = set(), []
if os.path.exists(OUT_CSV):
    prev = pd.read_csv(OUT_CSV)
    results = prev.to_dict("records")
    done = set(zip(prev["duration"], prev["stem"]))
    print("Бұрынғы нәтиже:", len(results), "жол, өткізіледі.")

t0 = time.time()
for item in tqdm(pairs, desc="MMS + VAD"):
    key = (item["duration"], item["stem"])
    if key in done: continue
    try:
        ref = open(item["ref_path"], encoding="utf-8").read().strip()
        raw = transcribe_mms_vad(item["audio_path"])
        m = all_metrics(ref, raw)
        row = dict(duration=item["duration"], stem=item["stem"], ref=ref, raw=raw, fixed=raw)
        row.update({f"{k}_raw": v for k, v in m.items()})
        results.append(row); done.add(key)
    except Exception as e:
        print("[ERROR]", item["duration"], item["stem"], e); continue
    pd.DataFrame(results).to_csv(OUT_CSV, index=False)   # ӘР файлдан кейін сақтау

df_mms = pd.DataFrame(results)
df_mms.to_csv(OUT_CSV, index=False)
print("Аяқталды:", len(df_mms), "/", len(pairs), "|", round((time.time()-t0)/60, 1), "мин")
print("Сақталды:", OUT_CSV)

MMS + VAD:   0%|          | 0/100 [00:00<?, ?it/s]

Аяқталды: 100 / 100 | 12.3 мин
Сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/mms/mapp_v2_batch_results_mms_vad.csv


In [ ]:
# Біркелкі нормалау (кіші әріп + тыныс белгісіз) бойынша MMS vs kk-turbo (мәтіннен қайта есептеледі)
def collapse_loops(text, max_run=2):
    out, prev, run = [], None, 0
    for w in str(text).split():
        if w == prev:
            run += 1
            if run <= max_run: out.append(w)
        else:
            prev, run = w, 1; out.append(w)
    return " ".join(out)

def mean_fast(df, col):
    ms = [all_metrics(r, h, with_ter=False, with_bert=False) for r, h in zip(df["ref"], df[col])]
    return pd.DataFrame(ms).mean()

cols = {"MMS+VAD": mean_fast(df_mms, "raw")}
for name, fn in [("kkturbo_chunk", "mapp_v2_batch_results_kkturbo_chunk.csv"),
                 ("kkturbo_vad", "mapp_v2_batch_results_kkturbo_vad.csv")]:
    pth = os.path.join(DATASET_BASE, fn)
    if os.path.exists(pth):
        d = pd.read_csv(pth)
        cols[name + " raw"] = mean_fast(d, "raw")
        d["raw_decloop"] = d["raw"].apply(collapse_loops)
        cols[name + "+decloop"] = mean_fast(d, "raw_decloop")
cmp_df = pd.DataFrame(cols).round(4)
display(cmp_df)

print("\nMMS толық 6 метрика (орташа):")
print(df_mms[[c for c in df_mms.columns if c.endswith("_raw") and c != "raw"]].mean().round(4))

OUT_XLSX = os.path.join(OUT_DIR, "mapp_asr_mms_comparison.xlsx")
_bad = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
clean = lambda v: _bad.sub("", v) if isinstance(v, str) else v
with pd.ExcelWriter(OUT_XLSX) as w:
    cmp_df.to_excel(w, sheet_name="fair_comparison")
    df_mms.map(clean).to_excel(w, sheet_name="mms_per_file", index=False)
print("Excel сақталды:", OUT_XLSX)

,MMS+VAD,kkturbo_chunk raw,kkturbo_chunk+decloop,kkturbo_vad raw,kkturbo_vad+decloop
wer,0.5412,0.4052,0.3383,0.5485,0.2975
cer,0.1936,0.2716,0.2246,0.3961,0.1704
bleu,27.1029,51.2900,52.6491,49.4298,55.9698
chrf,72.8043,78.3841,79.0182,82.7039,85.7228
ter,NaN,NaN,NaN,NaN,NaN
bertscore,NaN,NaN,NaN,NaN,NaN



MMS толық 6 метрика (орташа):
wer_raw           0.5412
cer_raw           0.1936
bleu_raw         27.1029
chrf_raw         72.8043
ter_raw          54.3102
bertscore_raw     0.8384
dtype: float64
Excel сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/mms/mapp_asr_mms_comparison.xlsx
